# Day model v3 — pruned features

Keep the columns v2 SHAP actually used: yesterday / last week / day-before (`lag_48`, `lag_336`, `lag_96`), wind and shortwave 1d+7d deltas, forward temperature, a short calendar set, and `lead_hours`.

Drop EMAs, 3-week demand extras, HDD/CDD stacks, hour/minute/month duplicates, and the 1d/7d deltas that did not show up. 22 columns vs 57 on v2 / 37 on production.

Leads 0–24h only. Writes to `notebooks/day/v3/models`. `PROMOTE = False`. Live scores every day on the board.

In [ ]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

ROOT = Path.cwd().resolve()
while not (ROOT / "src" / "energy_forecast").is_dir() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
HERE = ROOT / "notebooks" / "day" / "v3"
OUT_DIR = HERE / "models"
V1_DIR = ROOT / "notebooks" / "day" / "v1" / "models"
V2_DIR = ROOT / "notebooks" / "day" / "v2" / "models"
sys.path.insert(0, str(ROOT / "src"))

from energy_forecast import tune
from energy_forecast.data.demand import load_demand
from energy_forecast.evaluate import as_board_scored, board_naive_scale, overall_metrics, score
from energy_forecast.explain import explain_scored, mean_abs_shap, shap_source_share
from energy_forecast.features import (
    DAY_V1_FEATURES,
    DAY_V2_FEATURES,
    DAY_V3_FEATURES,
    FEATURE_COLUMNS,
    attach_lookback_demand,
    attach_lookback_weather,
)
from energy_forecast.forecast import prepare_history
from energy_forecast.live import DAY, load_board, load_context_actuals
from energy_forecast.model import (
    DAY_MAX_LEAD_HOURS,
    load_day_models,
    load_models,
    predict_quantiles,
    split_training_frame,
    train_models,
)
from energy_forecast.modes import SHORT, resample_to_mode
from energy_forecast.regions import SERIES_ID
from energy_forecast.review import (
    _attach_frozen_and_weather,
    _ensure_calendar,
    _load_archive_weather,
    _pack_rows,
    _ready_predict_frame,
    _recompute_weather_features,
)
from energy_forecast.settlement import LONDON_TZ
from energy_forecast.tune import naive_scale_for_product

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.float_format", lambda v: f"{v:,.1f}")
print("repo", ROOT)
print("write models to", OUT_DIR)
print("n features", len(DAY_V3_FEATURES), "vs production", len(FEATURE_COLUMNS), "vs v2", len(DAY_V2_FEATURES))
print("kept", DAY_V3_FEATURES)
prod = load_day_models()
print("production day features", len(prod.features))

## 1. History

Previous Runs, leads ≤ 24h.

In [ ]:
history = prepare_history()
train, holdout = split_training_frame(
    history, features=FEATURE_COLUMNS, max_lead_hours=DAY_MAX_LEAD_HOURS
)
_, holdout_v3 = split_training_frame(
    history, features=DAY_V3_FEATURES, max_lead_hours=DAY_MAX_LEAD_HOURS
)
scale = naive_scale_for_product(train, "short")
print("train", len(train), "holdout", len(holdout), "v3 holdout", len(holdout_v3))

## 2. Optuna

Hourly 24h holdout on `DAY_V3_FEATURES`. Study: `data/processed/optuna_day_v3.db`.

In [ ]:
PRODUCT = "day_v3"
N_TRIALS = 40
TRAIN_SAMPLE = None
USE_WANDB = False

tune_train, tune_holdout = tune.product_split(history, PRODUCT, train_sample=TRAIN_SAMPLE)
print(PRODUCT, "train", len(tune_train), "holdout", len(tune_holdout))

baseline = tune.baseline_params(PRODUCT)
print("baseline params", baseline)
base_models = tune.fit_trial(tune_train, PRODUCT, baseline, refit_full=False)
baseline_mae, baseline_metrics = tune.holdout_mae(tune_holdout, base_models, PRODUCT, train=tune_train)
print("baseline MAE", baseline_mae)
display(baseline_metrics)

study = tune.create_study(PRODUCT)
with tune.tracking_run(PRODUCT, study, wandb_on=USE_WANDB):
    study.optimize(tune.make_objective(tune_train, tune_holdout, PRODUCT), n_trials=N_TRIALS, show_progress_bar=True)
    tune.log_study_end(study, PRODUCT)
print("best MAE", study.best_value)
print("best params", study.best_params)
print("beats baseline", study.best_value < baseline_mae, "by", baseline_mae - study.best_value, "MW")

In [ ]:
rows = pd.DataFrame(tune.study_trial_rows(study, PRODUCT))
display(rows.sort_values("mae").head(10))
best_path = HERE / "optuna_best.json"
best_payload = {key: study.best_params[key] for key in tune.PARAM_KEYS if key in study.best_params}
best_payload["holdout_mae"] = float(study.best_value)
best_path.write_text(json.dumps(best_payload, indent=2), encoding="utf-8")
print("wrote", best_path)

## 3. Train day v3

Uses `optuna_best.json` if you ran section 2. Does not overwrite `data/models/day`.

In [ ]:
OUT_DIR.mkdir(parents=True, exist_ok=True)
best_path = HERE / "optuna_best.json"
lgb_params = None
if best_path.exists():
    raw = json.loads(best_path.read_text(encoding="utf-8"))
    lgb_params = {key: raw[key] for key in tune.PARAM_KEYS if key in raw}
    print("using Optuna winner", lgb_params)
else:
    print("no optuna_best.json; stock LightGBM")

v3, metrics = train_models(
    history,
    model_dir=OUT_DIR,
    role="next_day",
    features=DAY_V3_FEATURES,
    max_lead_hours=DAY_MAX_LEAD_HOURS,
    lgb_params=lgb_params,
)
meta_path = OUT_DIR / "metadata.json"
meta = json.loads(meta_path.read_text(encoding="utf-8"))
meta["pruned_day"] = True
if lgb_params:
    meta["optuna_params"] = lgb_params
meta_path.write_text(json.dumps(meta, indent=2), encoding="utf-8")
print("saved", OUT_DIR, "n features", len(v3.features))
metrics

## 4. Holdout (hourly, 24h)

In [ ]:
from energy_forecast.model import load_models as load_from

v3 = load_from(OUT_DIR)
v1 = load_from(V1_DIR) if (V1_DIR / "metadata.json").exists() else None
v2 = load_from(V2_DIR) if (V2_DIR / "metadata.json").exists() else None


def pack_metrics(frame, models, label):
    scored = resample_to_mode(score(frame, models), SHORT)
    out = overall_metrics(scored, scale_mse=scale)
    out.insert(0, "model", label)
    return out


rows = [pack_metrics(holdout, prod, "production day")]
if v1 is not None:
    _, holdout_v1 = split_training_frame(
        history, features=DAY_V1_FEATURES, max_lead_hours=DAY_MAX_LEAD_HOURS
    )
    rows.append(pack_metrics(holdout_v1, v1, "day v1 / 3w extras"))
if v2 is not None:
    _, holdout_v2 = split_training_frame(
        history, features=DAY_V2_FEATURES, max_lead_hours=DAY_MAX_LEAD_HOURS
    )
    rows.append(pack_metrics(holdout_v2, v2, "day v2 / yesterday"))
rows.append(pack_metrics(holdout_v3, v3, "day v3 / pruned"))
pd.concat(rows, ignore_index=True)

## 5. All completed live days

Every London day on the board with actuals.

In [ ]:
demand_hist = load_demand(include_live=True)


def shap_ready(frame, models):
    out = _ensure_calendar(frame.copy().reset_index(drop=True))
    out["region_id"] = SERIES_ID
    out["timestamp"] = pd.to_datetime(out["timestamp"], utc=True)
    keep_ts = out["timestamp"].copy()
    work = attach_lookback_weather(out.drop(columns=["window_start"], errors="ignore"), _load_archive_weather())
    work = attach_lookback_demand(work, demand_hist, days=22)
    rebuilt = _recompute_weather_features(work)
    keys = ["timestamp", "lead_hours"] if "lead_hours" in out.columns else ["timestamp"]
    take = [c for c in models.features if c in rebuilt.columns]
    right_keys = [k for k in keys if k in rebuilt.columns]
    right = rebuilt.loc[:, list(dict.fromkeys([*right_keys, *take]))].drop_duplicates(right_keys, keep="last")
    merged = out.merge(right, on=right_keys, how="left", suffixes=("", "_rebuilt"))
    for column in models.features:
        rebuilt_col = f"{column}_rebuilt"
        if column not in merged.columns and rebuilt_col in merged.columns:
            merged[column] = merged[rebuilt_col]
        elif rebuilt_col in merged.columns:
            merged[column] = pd.to_numeric(merged[column], errors="coerce").combine_first(
                pd.to_numeric(merged[rebuilt_col], errors="coerce")
            )
    merged["region_id"] = SERIES_ID
    merged = merged.loc[pd.to_datetime(merged["timestamp"], utc=True).isin(set(keep_ts))].copy()
    return _ready_predict_frame(merged, models)


board = load_board()
ctx = load_context_actuals()
joined = _attach_frozen_and_weather(
    DAY,
    _pack_rows(DAY, board, ctx, day_days=None).dropna(subset=["actual_mw", "p50"]),
    _load_archive_weather(),
)
joined["timestamp"] = pd.to_datetime(joined["timestamp"], utc=True)
joined["region_id"] = SERIES_ID
local = joined["timestamp"].dt.tz_convert(LONDON_TZ)
print("live days", local.dt.normalize().nunique(), "rows", len(joined), joined["timestamp"].min(), "->", joined["timestamp"].max())
scale_live = board_naive_scale()


def metrics_row(frame, label, climatology):
    scored = as_board_scored(frame, climatology=climatology)
    out = overall_metrics(scored, scale_mse=scale_live)
    out.insert(0, "model", label)
    return out, scored


def fill_hat(base, models):
    hat = predict_quantiles(shap_ready(base, models), models)
    days = base.copy()
    days[["p10", "p50", "p90"]] = hat[["p10", "p50", "p90"]].to_numpy()
    return days


issued_tbl, _ = metrics_row(joined, "issued (board)", prod.climatology)
prod_days = fill_hat(joined, prod)
prod_tbl, _ = metrics_row(prod_days, "production day (re-score)", prod.climatology)
live_tbls = [issued_tbl, prod_tbl]
hats = {"issued": joined, "prod": prod_days}
if v1 is not None:
    v1_days = fill_hat(joined, v1)
    hats["v1"] = v1_days
    v1_tbl, _ = metrics_row(v1_days, "day v1 / 3w extras", v1.climatology)
    live_tbls.append(v1_tbl)
if v2 is not None:
    v2_days = fill_hat(joined, v2)
    hats["v2"] = v2_days
    v2_tbl, _ = metrics_row(v2_days, "day v2 / yesterday", v2.climatology)
    live_tbls.append(v2_tbl)
v3_days = fill_hat(joined, v3)
hats["v3"] = v3_days
v3_tbl, _ = metrics_row(v3_days, "day v3 / pruned", v3.climatology)
live_tbls.append(v3_tbl)
pd.concat(live_tbls, ignore_index=True)

In [ ]:
payload = {
    "day": local.dt.strftime("%a %d %b"),
    "actual": pd.to_numeric(joined["actual_mw"], errors="coerce"),
    "issued": pd.to_numeric(joined["p50"], errors="coerce"),
    "prod": pd.to_numeric(prod_days["p50"], errors="coerce"),
    "v3": pd.to_numeric(v3_days["p50"], errors="coerce"),
}
for key in ("v1", "v2"):
    if key in hats:
        payload[key] = pd.to_numeric(hats[key]["p50"], errors="coerce")
daily = pd.DataFrame(payload)
rows = []
for day, g in daily.groupby("day", sort=False):
    row = {
        "day": day,
        "n": int(len(g)),
        "actual": float(g["actual"].mean()),
        "issued MAE": float((g["actual"] - g["issued"]).abs().mean()),
        "prod MAE": float((g["actual"] - g["prod"]).abs().mean()),
        "v3 MAE": float((g["actual"] - g["v3"]).abs().mean()),
    }
    if "v1" in g.columns:
        row["v1 MAE"] = float((g["actual"] - g["v1"]).abs().mean())
    if "v2" in g.columns:
        row["v2 MAE"] = float((g["actual"] - g["v2"]).abs().mean())
    rows.append(row)
pd.DataFrame(rows)

## 6. SHAP

v3 should be mostly `lag_48` / `lag_336` plus wind and shortwave deltas. No EMAs, no 3-week mean.

In [ ]:
n = 2_500
need_p = [c for c in prod.features if c in holdout.columns]
need_v = [c for c in v3.features if c in holdout_v3.columns]
prod_s = holdout.dropna(subset=["demand_mw", *need_p]).sample(n, random_state=0)
v3_s = holdout_v3.dropna(subset=["demand_mw", *need_v]).sample(n, random_state=0)
if "region_id" not in prod_s.columns:
    prod_s["region_id"] = SERIES_ID
if "region_id" not in v3_s.columns:
    v3_s["region_id"] = SERIES_ID
prod_shap = explain_scored(predict_quantiles(prod_s, prod), prod)["mean"]
v3_shap = explain_scored(predict_quantiles(v3_s, v3), v3)["mean"]
share = pd.concat(
    [
        shap_source_share(prod_shap).assign(model="production day"),
        shap_source_share(v3_shap).assign(model="day v3"),
    ],
    ignore_index=True,
)
print("Holdout |SHAP| share")
display(share.pivot(index="source", columns="model", values="share"))
print("v3 all features")
display(mean_abs_shap(v3_shap, n=len(v3.features)))

fig, axes = plt.subplots(1, 2, figsize=(10, 4.2))
for ax, shap_df, title in (
    (axes[0], prod_shap, "Production day"),
    (axes[1], v3_shap, "Day v3 pruned"),
):
    rank = mean_abs_shap(shap_df, n=10)
    ax.barh(rank["feature"].iloc[::-1], rank["mean_abs_shap"].iloc[::-1])
    ax.set_title(title)
    ax.set_xlabel("mean |SHAP| (MW)")
fig.tight_layout()

## 7. Promote (optional)

Copies day v3 over `data/models/day`. Week and hour folders are untouched.

In [ ]:
PROMOTE = False
if PROMOTE:
    from energy_forecast.paths import DATA_MODELS_DAY
    import shutil

    DATA_MODELS_DAY.mkdir(parents=True, exist_ok=True)
    for src in OUT_DIR.glob("*"):
        if src.is_file():
            shutil.copy2(src, DATA_MODELS_DAY / src.name)
    print("promoted day artefacts to", DATA_MODELS_DAY)
else:
    print("skipped promote; artefacts remain in", OUT_DIR)